<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/SFDS/dragon_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Royal Draconic Conservatory - Regression**

# **Importing Libraries**

In [1]:
!pip -q install plotly scikit-learn tabulate pandas

In [2]:
# Libraries for data loading, data manipulation and data visulisation
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import rc
import seaborn as sns
import plotly.express as px
from scipy.stats import kurtosis


#Feature engineering, selection and Model training
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


#ignoring warnings
import warnings
warnings.filterwarnings('ignore')

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
from tabulate import tabulate
from IPython.display import display


# **Data acquisition**

In [4]:
# Loading Data
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/SFDS/dragon_data.csv"
df = pd.read_csv(url)
df.head()

,SPC,AGE,MASS,WSP,HID,SPD,FHO
0,Wyvern,16,1304,4.0,4.83,4800,611
1,Dragon,2726,3693174,52.0,4.82,114400,1960
2,Dragon,377,154201,19.4,4.61,42680,1123
3,Hydra,169,40146,13.0,2.65,7800,706
4,Dragon,1571,1549315,39.6,1.49,87120,1441


In [5]:
# Create the data as a list of dictionaries
variables = [

    {"variable": "FHO", "description": "The target variable — Flame Heat Output (in degrees Celsius)."},
    {"variable": "WSP", "description": "Wingspan (in meters)."},
    {"variable": "MASS", "description": "Mass (in metric tons)."},
    {"variable": "AGE", "description": "Age (in years)."},
    {"variable": "SPD", "description": "Sustained flight speed (in km/h)."},
    {"variable": "HID", "description": "Hide thickness (in cm)."},
    {"variable": "SPC", "description": "Species (e.g. Dragon, Wyvern or Hyrdra)."},

]

# Display the DataFrame
var_desc = pd.DataFrame(variables)
var_desc_final = var_desc.reset_index(drop = True)

# Display the DataFrame
print(tabulate(var_desc_final, headers = 'keys', tablefmt = 'grid'))

+----+------------+---------------------------------------------------------------+
|    | variable   | description                                                   |
+====+============+===============================================================+
|  0 | FHO        | The target variable — Flame Heat Output (in degrees Celsius). |
+----+------------+---------------------------------------------------------------+
|  1 | WSP        | Wingspan (in meters).                                         |
+----+------------+---------------------------------------------------------------+
|  2 | MASS       | Mass (in metric tons).                                        |
+----+------------+---------------------------------------------------------------+
|  3 | AGE        | Age (in years).                                               |
+----+------------+---------------------------------------------------------------+
|  4 | SPD        | Sustained flight speed (in km/h).                       

# **Data Transformation**

- This deals with the skewness of these two features.

In [ ]:
# initialise encoder
le = LabelEncoder()

# Encoding Nest feature
df_semi['nest_encoded'] = le.fit_transform(df_semi['nest'])

# Encoding target variable
df_semi['spec_encoded'] = le.fit_transform(df_semi['spec'])

- LabelEncoder() provides a simple, consistent, and lossless way to convert variables to numeric because most machine learning algorithms can only work with numbers, not text or categories.

In [ ]:
# initialise scaler
scaler = StandardScaler()

# scale age using standard scaler
df_semi['age_scaled'] = scaler.fit_transform(df_semi[['age']])

# scale age using standard scaler
df_semi['temp_scaled'] = scaler.fit_transform(df_semi[['temp']])

- StandardScaler standardize numeric features so they have mean 0 and variance 1, ensuring all features contribute equally and improving model performance.

In [ ]:
# binning spot feature
df_semi['spot_bin'] = np.where(
    df_semi['spot'] <= 5, 0,
    np.where(df_semi['spot'] <= 10, 1, 3)
)

- Binning is useful because it simplifies continuous data into categories, reducing noise, and helping models handle non-linear relationships.

In [ ]:
df_semi.head()

,spec,vol,mass,age,spot,nest,temp,spc,density,nest_encoded,spec_encoded,age_scaled,temp_scaled,spot_bin
0,Wyvern,398,665,446,10,Coastal,38.7,Wyvern,1.085092,0,2,1.348756,-0.098936,1
1,Hydra,13,7,477,8,Volcanic,82.9,Hydra,0.767874,4,1,1.564454,2.102293,1
2,Hydra,61,32,74,8,Coastal,29.0,Hydra,0.843928,0,1,-1.239619,-0.582011,1
3,Hydra,44,68,207,9,Forest,21.8,Hydra,1.105903,1,1,-0.314205,-0.940582,1
4,Dragon,99,120,61,2,Volcanic,70.9,Dragon,1.039147,4,0,-1.330073,1.504674,0


In [ ]:
df_semi.columns

Index(['spec', 'vol', 'mass', 'age', 'spot', 'nest', 'temp', 'spc', 'density',
       'nest_encoded', 'spec_encoded', 'age_scaled', 'temp_scaled',
       'spot_bin'],
      dtype='str')

In [ ]:
#  final dataframe
df_final = df_semi[['density','nest_encoded','age_scaled','temp_scaled','spot_bin','spec_encoded']]

# Rename specific columns
df_final = df_final.rename(columns={
    'nest_encoded': 'nest',
    'age_scaled': 'age',
    'temp_scaled': 'temp',
    'spot_bin': 'spot',
    'spec_encoded': 'spec'
})

In [ ]:
df_final.head()

,density,nest,age,temp,spot,spec
0,1.085092,0,1.348756,-0.098936,1,2
1,0.767874,4,1.564454,2.102293,1,1
2,0.843928,0,-1.239619,-0.582011,1,1
3,1.105903,1,-0.314205,-0.940582,1,1
4,1.039147,4,-1.330073,1.504674,0,0


In [ ]:
df_final.spec.value_counts()

spec
1    243
2    166
0     61
Name: count, dtype: int64

In [ ]:
df_main.spec.value_counts()

spec
Hydra     260
Wyvern    173
Dragon     67
Name: count, dtype: int64

#### Data Split
Splitting data into training and testing sets is essential to evaluate a model's performance on new, unseen data and prevent overfitting. The training set teaches the model patterns, while the untouched test set provides an unbiased assessment of its generalisation ability, ensuring the model performs accurately in real-world scenarios rather than just on the data it learned from.


In [ ]:
# features and target variable
X = df_final.drop(['spec'], axis = 1)
y = df_final['spec']

In [ ]:
# Split into train+val and test (80/20)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Split train+val into train and validation (75/25 of trainval = 60/20 overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

# **Single Variable Regression**



In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

# Classification report
report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)

Classification Report:
               precision    recall  f1-score   support

           0       0.17      0.08      0.11        12
           1       0.64      0.78      0.70        49
           2       0.66      0.58      0.61        33

    accuracy                           0.62        94
   macro avg       0.49      0.48      0.48        94
weighted avg       0.59      0.62      0.60        94



In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[ 1  9  2]
 [ 3 38  8]
 [ 2 12 19]]


# **Multiple Variable Regression**



In [ ]:
# Evaluation Results
#print(tabulate(results_df, headers='keys', tablefmt='grid'))